In [1]:
import kymnasium as kym
import gymnasium as gym


env = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='none'
)
env.reset()

({'mario': array([288., 768., 336., 816.,   0.], dtype=float32),
  'blurps': array([[0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.

In [2]:
import numpy as np


class ReplayBuffer:
    def __init__(
            self,
            state_dim: int,
            state_seq: int,
            capacity: int,
            alpha: float,
            beta: float,
            seed: int | None = None,
    ):
        self._alpha = alpha
        self._beta = beta
        self._capacity = capacity
        self._state_dim = state_dim
        self._state_seq = state_seq

        self._states = np.zeros(shape=(self._capacity, self._state_seq, self._state_dim), dtype='float32')
        self._actions = np.zeros(shape=(self._capacity, ), dtype='int32')
        self._rewards = np.zeros(shape=(self._capacity,), dtype='float32')
        self._next_states = np.zeros(shape=(self._capacity, self._state_seq, self._state_dim), dtype='float32')
        self._dones = np.zeros(shape=(self._capacity,), dtype='float32')
        self._priorities = np.ones(shape=(self._capacity,), dtype='float32')

        self._max_priority = 1.0
        self._size = 0
        self._index = 0

        self._random = np.random.default_rng(seed)

    def add(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool):
        self._states[self._index] = state
        self._actions[self._index] = action
        self._rewards[self._index] = reward
        self._next_states[self._index] = next_state
        self._dones[self._index] = done
        self._priorities[self._index] = self._max_priority
        self._index = (self._index + 1) % self._capacity
        self._size = min(self._size + 1, self._capacity)

    def sample(self, batch_size: int) -> tuple | None:
        if self._size < batch_size:
            return None

        probs = self._priorities[:self._size] ** self._alpha
        probs /= probs.sum()

        indices = self._random.choice(self._size, size=batch_size, p=probs, replace=False)

        weights = (self._size * probs[indices]) ** (-self._beta)
        weights /= weights.max()
        weights = np.array(weights, dtype=np.float32)

        return (
            self._states[indices],
            self._actions[indices],
            self._rewards[indices],
            self._next_states[indices],
            self._dones[indices],
            indices,
            weights
        )

    def update_priorities(self, indices: np.ndarray, priorities: np.ndarray):
        max_priority = priorities.max()
        self._max_priority = max(self._max_priority, max_priority)

        for index, priority in zip(indices, priorities):
            self._priorities[index] = float(priority) + 1e-8

In [3]:
from tensorflow import keras


def build_network(input_shape, n_action):
    inputs = keras.Input(shape=input_shape)

    x = keras.layers.LayerNormalization()(inputs)
    x = keras.layers.GRU(
        128,
        return_sequences=False,
        reset_after=True,
        recurrent_initializer="orthogonal",
    )(x)

    x = keras.layers.Dense(
        128,
        activation="relu",
        kernel_initializer="he_normal",
    )(x)

    value = keras.layers.Dense(
        128,
        activation="relu",
        kernel_initializer="he_normal",
    )(x)
    value = keras.layers.Dense(1)(value)

    advantage = keras.layers.Dense(
        128,
        activation="relu",
        kernel_initializer="he_normal",
    )(x)
    advantage = keras.layers.Dense(n_action)(advantage)

    q_values = value + advantage - keras.ops.mean(
        advantage,
        axis=1,
        keepdims=True,
    )

    model = keras.Model(inputs=inputs, outputs=q_values)
    return model

I0000 00:00:1779251359.113781 3704438 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import kymnasium as kym
import tensorflow as tf
from tensorflow import keras
from collections import deque
import json
import os


class Agent(kym.Agent):
    def __init__(
            self,
            behavior_network: keras.models.Model,
            target_network: keras.models.Model,
            width: int,
            height: int,
            state_dim: int,
            state_seq: int,
            action_space: int,
            buffer_size: int,
            alpha: float,
            beta: float,
            gamma: float,
            tau: float,
            learning_rate: float,
            batch_size: int,
            episode_monitor: int,
            seed: int | None = None,
    ):
        self._behavior_network = behavior_network
        self._target_network = target_network
        self._width = width
        self._height = height
        self._state_seq = state_seq
        self._state_dim = state_dim
        self._action_space = action_space
        self._buffer_size = buffer_size
        self._alpha = alpha
        self._beta = beta
        self._gamma = gamma
        self._tau = tau
        self._learning_rate = learning_rate
        self._batch_size = batch_size
        self._episode_monitor = episode_monitor
        self._seed = seed

        self._target_network.set_weights(self._behavior_network.get_weights())

        self._replay_buffer = ReplayBuffer(
            state_dim=self._state_dim,
            state_seq=self._state_seq,
            capacity=buffer_size,
            alpha=alpha,
            beta=beta,
            seed=self._seed,
        )
        self._optimizer = keras.optimizers.Adam(learning_rate=self._learning_rate, clipnorm=1.0)
        self._objective = keras.losses.Huber(reduction=None)

        self._losses = deque(maxlen=self._episode_monitor)
        self._rewards = deque(maxlen=self._episode_monitor)
        self._action_counts = [
            deque(maxlen=self._episode_monitor) for _ in range(self._action_space)
        ]
        self._elapsed_time = deque(maxlen=self._episode_monitor)

        self._running_state: deque[np.ndarray] = deque(maxlen=self._state_seq)

    @property
    def loss_(self):
        return np.mean(self._losses) if len(self._losses) > 0 else 0.0

    @property
    def reward_(self):
        return np.mean(self._rewards) if len(self._rewards) > 0 else 0.0

    @property
    def action_counts_(self):
        return [
            np.mean(self._action_counts[i]) if len(self._action_counts[i]) > 0 else 0.0
            for i in range(self._action_space)
        ]

    @property
    def elapsed_time_(self):
        return np.mean(self._elapsed_time) if len(self._elapsed_time) > 0 else 0.0

    def to_state(self, obs: dict) -> np.ndarray:
        mario, blurps = obs['mario'], obs['blurps']
        mario = np.array(
            [mario[0] / self._width, mario[1] / self._height, mario[-1] / self._width],
            dtype=np.float32
        )
        blurps = np.array([
            [blurp[0] / self._width, blurp[1] / self._height, blurp[4] / self._width, blurp[5] / self._height, blurp[6] / self._height]
            for blurp in blurps
        ], dtype=np.float32)
        blurps = np.reshape(blurps, -1)

        self._running_state.append(np.concatenate([mario, blurps]))
        return np.asarray(self._running_state)

    def add(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool):
        self._replay_buffer.add(
            state, action, reward, next_state, done
        )

    def act(self, obs: np.ndarray, info: dict):
        state = keras.ops.expand_dims(self.to_state(obs), axis=0)
        q = self._behavior_network(state, training=False)
        q = keras.ops.ravel(q)
        return keras.ops.argmax(q).numpy()

    @tf.function
    def choose_action(self, state: keras.KerasTensor):
        state = keras.ops.expand_dims(state, axis=0)
        q = self._behavior_network(state, training=False)
        q = keras.ops.ravel(q)
        return keras.ops.argmax(q)

    @tf.function
    def train(self, states, actions, rewards, next_states, dones, weights):
        next_actions = keras.ops.argmax(
            self._behavior_network(next_states, training=False),
            axis=1
        )
        next_actions = keras.ops.one_hot(next_actions, self._action_space)

        next_action_values = keras.ops.sum(
            self._target_network(next_states, training=False) * next_actions,
            axis=1,
            keepdims=True
        )

        targets = rewards + (1 - dones) * self._gamma * next_action_values
        targets = tf.stop_gradient(targets)

        with tf.GradientTape() as tape:
            action_values = keras.ops.sum(
                self._behavior_network(states, training=True) * actions,
                axis=1,
                keepdims=True
            )
            per_sample_loss = self._objective(action_values, targets)
            per_sample_loss = keras.ops.expand_dims(per_sample_loss, axis=1)
            loss = keras.ops.mean(per_sample_loss * weights)

            priorities = keras.ops.abs(targets - action_values) + 1e-8

        gradients = tape.gradient(loss, self._behavior_network.trainable_variables)
        self._optimizer.apply_gradients(zip(gradients, self._behavior_network.trainable_variables))

        return loss, priorities

    def replay(self):
        batches = self._replay_buffer.sample(self._batch_size)
        if batches is None:
            return

        states, actions, rewards, next_states, dones, indices, weights = batches

        loss, priorities = self.train(
            states=keras.ops.convert_to_tensor(states),
            actions=keras.ops.one_hot(actions.astype(np.int32), self._action_space),
            rewards=keras.ops.expand_dims(rewards, axis=1),
            next_states=keras.ops.convert_to_tensor(next_states),
            dones=keras.ops.expand_dims(dones, axis=1),
            weights=keras.ops.expand_dims(weights, axis=1)
        )

        priorities = priorities.numpy().reshape(-1)
        self._replay_buffer.update_priorities(indices, priorities)

        self._losses.append(loss)

        for target_var, behavior_var in zip(
            self._target_network.trainable_variables,
            self._behavior_network.trainable_variables,
        ):
            target_var.assign(
                self._tau * behavior_var + (1.0 - self._tau) * target_var
            )

    def begin_episode(self):
        self._running_state.clear()

        for _ in range(self._state_seq):
            self._running_state.append(np.zeros(self._state_dim))

    def end_episode(self, rewards, actions, elapsed_time):
        self._rewards.append(rewards)

        for i in range(self._action_space):
            self._action_counts[i].append(actions[i])

        self._elapsed_time.append(elapsed_time)

    def save(self, path: str):
        config = dict(
            width=self._width,
            height=self._height,
            state_dim=self._state_dim,
            state_seq=self._state_seq,
            action_space=self._action_space,
            buffer_size=self._buffer_size,
            alpha=self._alpha,
            beta=self._beta,
            gamma=self._gamma,
            tau=self._tau,
            learning_rate=self._learning_rate,
            batch_size=self._batch_size,
            episode_monitor=self._episode_monitor,
            seed=self._seed,
        )
        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, 'config.json'), "w") as file:
            json.dump(config, file)

        keras.models.save_model(self._behavior_network, os.path.join(path, 'behavior_network.keras'))
        keras.models.save_model(self._target_network, os.path.join(path, 'target_network.keras'))

    @classmethod
    def load(cls, path: str) -> 'Agent':
        with open(os.path.join(path, 'config.json')) as file:
            config = json.load(file)
        behavior_network = keras.models.load_model(os.path.join(path, 'behavior_network.keras'))
        target_network = keras.models.load_model(os.path.join(path, 'target_network.keras'))

        return Agent(
            behavior_network = behavior_network,
            target_network = target_network,
            **config
        )

    def set_params(self, **kwargs):
        for key, value in kwargs.items():
            key = f'_{key}'
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                raise AttributeError(f"'Agent' object has no attribute '{key}'")

In [8]:
def box_gap_distance(box_a, box_b) -> float:
    al, at, ar, ab = box_a
    bl, bt, br, bb = box_b

    dx = max(bl - ar, al - br, 0.0)
    dy = max(bt - ab, at - bb, 0.0)

    return float((dx * dx + dy * dy) ** 0.5)


def shape_reward(next_obs: dict, cleared: bool, dead: bool) -> float:
    if cleared:
        return 100.0

    if dead:
        return -20.0

    reward = 0.02

    mario = next_obs["mario"]
    blurps = next_obs["blurps"]

    ml, mt, mr, mb, _ = mario
    mario_box = (ml, mt, mr, mb)

    nearest_gap = None
    immediate_lane_danger = False

    for blurp in blurps:
        bl, bt, br, bb, bvx, bvy, bg = blurp

        if br <= 0.0 and bb <= 0.0:
            continue

        blurp_box = (bl, bt, br, bb)
        gap = box_gap_distance(mario_box, blurp_box)

        if nearest_gap is None or gap < nearest_gap:
            nearest_gap = gap

        horizontal_overlap = not (br < ml or bl > mr)
        vertically_relevant = bt < mb + 192.0

        if horizontal_overlap and vertically_relevant:
            immediate_lane_danger = True

    if nearest_gap is None:
        return reward

    if nearest_gap < 24.0:
        reward -= 0.30
    elif nearest_gap < 48.0:
        reward -= 0.15
    elif nearest_gap < 96.0:
        reward -= 0.05
    elif nearest_gap < 192.0:
        reward += 0.005
    else:
        reward += 0.02

    if immediate_lane_danger:
        reward -= 0.10

    return float(reward)

In [12]:
from tqdm.auto import tqdm


def learn(
        max_episodes: int,
        steps_exploration: int,
        steps_epsilon_decay: int,
        steps_update_interval: int,
        epsilon_max: float,
        epsilon_min: float,
        episode_save_interval: int,
        agent: Agent,
        random_state: int|None = None,
        path: str|None = None,
):
    pbar = tqdm(range(max_episodes), desc='episode')
    random = np.random.default_rng(random_state)

    global_steps = 0
    epsilon = epsilon_max

    for episode in pbar:
        steps, total_reward, action_counts = 0, 0.0, np.zeros(3)

        agent.begin_episode()

        done = False
        obs, info = env.reset()
        state = agent.to_state(obs)
        while not done:
            if global_steps >= steps_exploration:
                progress = min(1.0, (global_steps - steps_exploration) / steps_epsilon_decay)
                epsilon = epsilon_max + progress * (epsilon_min - epsilon_max)

            if random.random() < epsilon:
                action = int(random.choice(3))
            else:
                action = int(agent.choose_action(state).numpy())

            next_obs, _, cleared, dead, info = env.step(action)
            next_state = agent.to_state(next_obs)
            reward = shape_reward(next_obs, cleared, dead)
            done = cleared or dead

            agent.add(state, action, reward, next_state, done)

            if global_steps % steps_update_interval == 0:
                agent.replay()

            state = next_state

            total_reward += reward
            global_steps += 1
            action_counts[action] += 1

        if action_counts.sum() > 0:
            action_counts = action_counts / action_counts.sum()
        agent.end_episode(total_reward, action_counts, info['time_elapsed'])

        if episode % episode_save_interval == 0 and path is not None:
            agent.save(os.path.join(path, f'Ep #{episode}'))

        a1, a2, a3 = agent.action_counts_
        pbar.set_postfix({
            'loss': f'{agent.loss_:.5f}',
            'reward': f'{agent.reward_:.5f}',
            'action=0': f'{a1:.5f}',
            'action=1': f'{a2:.5f}',
            'action=2': f'{a3:.5f}',
            'epsilon': f'{epsilon:.5f}',
            'elapsed': f'{agent.elapsed_time_:.3f}s'
        })

In [13]:
ddqn_agent = Agent(
    behavior_network=build_network((32, 3 + 30 * 5 ), 3),
    target_network=build_network((32, 3 + 30 * 5 ), 3),
    width=624,
    height=912,
    state_dim=3 + 30 * 5,
    state_seq=32,
    action_space=3,
    buffer_size=100000,
    alpha=0.6,
    beta=0.4,
    gamma=0.995,
    tau=0.005,
    learning_rate=0.00025,
    batch_size=64,
    episode_monitor=200,
    seed=42
)

learn(
    max_episodes=1_000_000,
    steps_exploration=50_000,
    steps_epsilon_decay=5_000_000,
    epsilon_min=0.05,
    episode_save_interval=1000,
    steps_update_interval=4,
    agent=ddqn_agent,
    random_state=42,
    path="/home/p4clab/Workspaces/avoid_blurps"
)

TypeError: learn() missing 1 required positional argument: 'epsilon_max'

In [14]:
agent = Agent.load('/home/p4clab/Workspaces/avoid_blurps/Ep #21000')


learn(
    max_episodes=1_000_000,
    steps_exploration=0,
    steps_epsilon_decay=2_500_000,
    epsilon_max=0.5,
    epsilon_min=0.05,
    episode_save_interval=1000,
    steps_update_interval=4,
    agent=agent,
    random_state=42,
    path="/home/p4clab/Workspaces/avoid_blurps"
)

episode:   0%|          | 0/1000000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
import kymnasium as kym


agent = Agent.load('./avoid_blurp/Ep #10000')
env_eval = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='human'
)

obs, info = env_eval.reset()
state = agent.to_state(obs)
done = False

while not done:
    action = agent.act(obs, info)
    next_obs, reward, terminated, truncated, info = env_eval.step(action)
    done = terminated or truncated
    obs = next_obs

In [43]:
import gymnasium as gym
import numpy as np


a = gym.spaces.Dict({
            'turn': gym.spaces.Discrete(2),
            'index': gym.spaces.Discrete(3),
            'power': gym.spaces.Box(0.0, 1.0, shape=(1,), dtype=np.float32),
            'angle': gym.spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32),
        })

a.sample()

{'angle': array([0.50650626], dtype=float32),
 'index': np.int64(1),
 'power': array([0.92182654], dtype=float32),
 'turn': np.int64(1)}

In [45]:
np.asarray([10]).item()

10